In [ ]:
import os
import json
import sys

import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import seaborn as sns

from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# sys.path.insert(1, '../scripts/')
# from reaches import *

In [ ]:
def readNHD(index, segmented=False):
    '''
    This function takes the index for an NHD HUC4 basin (see
    ./data/HUC4_lookup_no_great_lakes_PIXC.csv), reads it in,
    forces the geometry to 2D, and returns the basin dataframe.
    '''
    
    ## Set-up
    data_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/'
    mdata_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
    if segmented == False:
        print('type: normal')
        prep_path = f'{data_dir}NHD_prepped/'
    else:
        print('type: segmented')
        prep_path = f'{data_dir}NHD_prepped_segmented/'

    # Define dtypes for lookup tables to preserve leading zeros
    dtype_dic= {'HUC4': str, 'HUC2': str, 'toBasin': str, 'level': str}
    # Read in HUC lookup table
    lookup = pd.read_csv(os.path.join(mdata_dir,
                                      'HUC4_lookup_no_great_lakes.csv'),
                         dtype=dtype_dic)

    # Get current HUC2 and HUC4 IDs
    huc2 = 'HUC2_' + lookup.loc[index,'HUC4'][0:2]
    huc4 = 'NHDPLUS_H_' + lookup.loc[index,'HUC4'] + '_HU4_GDB'
    print(str(huc4))

    if segmented == False:
        # Set data filepath
        file_path = os.path.join(prep_path, huc2, huc4 + '_prepped.parquet')

        ## Read in prepped NHD flowlines
        features = ['NHDPlusID', 'GNIS_Name', 'LengthKM', 'WidthM',
                    'WidthM_Min', 'WidthM_Max', 'Bin', 'Bin_Min',
                    'Bin_Max', 'StreamOrde', 'Slope', 'Azimuth', 
                    'Sinuosity', 'geometry']
        
        basin = gpd.read_parquet(path=file_path, columns=features)
        # print('flowlines read-in')

        # Make geometry 2D LineStrings
        basin['geometry'] = basin.geometry.explode().force_2d()
        # print('exploded')

    else:
        # Set data filepath
        file_path = os.path.join(prep_path, huc2, huc4 + '_prepped_segmented.parquet')
        # print(file_path)

        ## Read in prepped NHD flowlines
        basin = gpd.read_parquet(path=file_path)
        # print('segments read-in')
        
    return basin, huc4, huc2

In [ ]:
def open_json(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
        return data

## Setup

In [ ]:
mdata_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'

In [ ]:
dtype_dic = {'tile': str, 'huc4': str, 'coverage': float}

In [ ]:
lookup = pd.read_csv(os.path.join(mdata_path,
                                  'huc4_swot_science_tiles.csv'),
                     dtype=dtype_dic)

In [ ]:
# # Get tiles in HUC2_17
# huc2 = '01' # '01' '03' '15' '17'
# huc2_01 = lookup[lookup['huc4'].str.startswith(huc2)]

# huc2 = '03' # '01' '03' '15' '17'
# huc2_03 = lookup[lookup['huc4'].str.startswith(huc2)]

# huc2 = '15' # '01' '03' '15' '17'
# huc2_15 = lookup[lookup['huc4'].str.startswith(huc2)]

# huc2 = '17' # '01' '03' '15' '17'
# huc2_17 = lookup[lookup['huc4'].str.startswith(huc2)]

In [ ]:
# palette_dict = {
#     '01': sns.color_palette('tab20b')[19],
#     '03': sns.color_palette('tab20b')[5],
#     '15': sns.color_palette('tab20b')[8],
#     '17': sns.color_palette('tab20b')[2]
# }

In [ ]:
# palette_letter_dict = {
#     'NE': sns.color_palette('tab20b')[19],
#     'SE': sns.color_palette('tab20c')[5],
#     'SW': sns.color_palette('tab20b')[8],
#     'NW': '#009E73'
# }

# palette_letter_dict = {
#     'NE': sns.color_palette('tab20b')[19],
#     'SE': '#009E73',
#     'SW': sns.color_palette('tab20b')[9],
#     'NW': sns.color_palette('tab20b')[1]
# }

# palette_letter_dict = {
#     'NE': '#CC79A7',
#     'SE': '#F0E442',
#     'SW': '#D55E00',
#     'NW': '#0072B2'
# }

# palette_letter_dict = {
#     'NE': sns.color_palette('tab20c')[1],
#     'SE': sns.color_palette('tab20c')[8],
#     'SW': sns.color_palette('tab20c')[4],
#     'NW': sns.color_palette('tab20c')[-3]
# }

palette_letter_dict = {
    'NE': sns.color_palette('tab20c')[1],
    'SE': sns.color_palette('tab20c')[4],
    'SW': sns.color_palette('tab20b')[9],
    # 'NW': sns.color_palette('tab20b')[-3]
    'NW': '#4C6AD1'
}

In [ ]:
# Get NHD index metadata
# Define dtypes for lookup tables to preserve leading zeros
dtype_dic= {'HUC4': str, 'HUC2': str, 'toBasin': str, 'level': str}
# Read in HUC lookup table
huc_lookup = pd.read_csv(os.path.join(mdata_path,
                                  'HUC4_lookup_no_great_lakes.csv'),
                     dtype=dtype_dic)

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]']

In [ ]:
# Read in matched IDs between Merritt and me
match_ids = open_json('./data/harlan_bennitt_match_ids.json')
# filter_ids = open_json('./data/filtered_ids.json')
post_ids = open_json('./data/any_data_post_ids.json')

In [ ]:
len(post_ids)

In [ ]:
# # Get high-quality IDs
# high_qual_ids = open_json('/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/high_qual_ids.json')

In [ ]:
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/figures/'

## Get HUCs

In [ ]:
# Read in HUC4 boundaries (no Great Lakes)
data_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/all_wbd_no_great_lakes.parquet'
wbd = gpd.read_parquet(path=data_path)

In [ ]:
wbd.crs

In [ ]:
wbd_ne = wbd[wbd.huc4.str.startswith('01')]
wbd_se = wbd[wbd.huc4.str.startswith('03')]
wbd_sw = wbd[wbd.huc4.str.startswith('15')]
wbd_nw = wbd[wbd.huc4.str.startswith('17')]

In [ ]:
wbd_combo = pd.concat([wbd_ne, wbd_se, wbd_sw, wbd_nw])

In [ ]:
hucs_01 = wbd[wbd['huc4'].str.startswith('01')]['huc4'].to_list()
hucs_03 = wbd[wbd['huc4'].str.startswith('03')]['huc4'].to_list()
hucs_15 = wbd[wbd['huc4'].str.startswith('15')]['huc4'].to_list()
hucs_17 = wbd[wbd['huc4'].str.startswith('17')]['huc4'].to_list()

In [ ]:
# Extract indices for read-in
indices_01 = list(huc_lookup[huc_lookup['HUC4'].isin(hucs_01)]['slurm_index'])
indices_03 = list(huc_lookup[huc_lookup['HUC4'].isin(hucs_03)]['slurm_index'])
indices_15 = list(huc_lookup[huc_lookup['HUC4'].isin(hucs_15)]['slurm_index'])
indices_17 = list(huc_lookup[huc_lookup['HUC4'].isin(hucs_17)]['slurm_index'])

### Get gauges

In [ ]:
# Table of all gauges with NHDPlus HR reaches mapped
nhd_sites = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata.parquet',
                            columns=['site_id', 'name', 'geometry'])

masks = pd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_combined.parquet',
                         columns=['site_id', 'NHDPlusID', 'WidthM', 'huc2', 'huc4', 'new_width'])

In [ ]:
nhd_sites = nhd_sites.to_crs(epsg=4269)
# nhd_sites.geometry = nhd_sites.geometry.map(lambda geom: shapely.ops.transform(lambda x, y: (y, x), geom))

In [ ]:
# Merge on names
nhd_sites = nhd_sites.merge(masks, on='site_id')

In [ ]:
mdata_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
site_ids = open_json(os.path.join(mdata_dir, 'usgs_gauges_cleaned_20251206.json'))

In [ ]:
nhd_sites = nhd_sites[nhd_sites['site_id'].isin(site_ids)]

# Remove reaches on rivers < 10 m wide
nhd_sites = nhd_sites[nhd_sites['WidthM'] >= 10]

assert nhd_sites.shape[0] == 2781

In [ ]:
# nhd_sites[nhd_sites['name'].str.contains('SNOQ')]

In [ ]:
# nhd_sites.iloc[[2964]].plot()

In [ ]:
# nhd_sites[nhd_sites['WidthM'] < 50]

### Get CONUS boundary

In [ ]:
exclude_states = ['Alaska', 'American Samoa',
                  'Commonwealth of the Northern Mariana Islands', 'Guam',
                  'District of Columbia', 'Puerto Rico',
                  'United States Virgin Islands', 'Hawaii']

In [ ]:
# CONUS boundary
states = gpd.read_file('/nas/cee-ice/data/NHDPlus_HR/other_shapefiles/cb_2018_us_state_5m.shp')
# states = gpd.read_file('/nas/cee-ice/data/NHDPlus_HR/other_shapefiles/cb_2018_us_nation_20m.shp')

states = states[~states['NAME'].isin(exclude_states)]
states = states.dissolve()

assert states.crs == 'EPSG:4269'

### Remove dangling hucs

In [ ]:
wbd = wbd.sjoin(states)

### Plot

In [ ]:
# Functions to format the numbers into "120° W" etc.
def lon_format(x, pos):
    return f"{abs(x):.0f}° W" if x < 0 else f"{x:.0f}° E"

def lat_format(y, pos):
    return f"{y:.0f}° N" if y > 0 else f"{abs(y):.0f}° S"

In [ ]:
# AREA PLOT
fig, ax = plt.subplots(figsize=(20,20))

wbd.plot(color=
         # '#F5F0E5',
        '#E8E8E8',
         ax=ax, edgecolor='white', linewidth=0.5)

wbd_ne.plot(color=palette_letter_dict['NE'], ax=ax,
            edgecolor='white', linewidth=0.5, label='NE');
wbd_se.plot(color=palette_letter_dict['SE'], ax=ax,
            edgecolor='white', linewidth=0.5, label='SE');
wbd_sw.plot(color=palette_letter_dict['SW'], ax=ax,
            edgecolor='white', linewidth=0.5, label='SW');
wbd_nw.plot(color=palette_letter_dict['NW'], ax=ax,
            edgecolor='white', linewidth=0.5, label='NE');

states.plot(color='none', edgecolor='dimgray', ax=ax, linewidth=1)

# 1. Create the L-frame (Remove top/right)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 2. Apply the custom degree formatters
ax.xaxis.set_major_formatter(FuncFormatter(lon_format))
ax.yaxis.set_major_formatter(FuncFormatter(lat_format))

# 3. Clean up ticks
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

plt.savefig(f'{save_dir}area_map.png', dpi=300, bbox_inches='tight')
# plt.savefig(os.path.join(save_dir, 'area_map.svg'), format='svg', bbox_inches='tight')

In [ ]:
# # GAUGES PLOT
# fig, ax = plt.subplots(figsize=(20,20))

# wbd.plot(color='#F5F0E5', ax=ax,
#         edgecolor='white', linewidth=0.5)

# states.plot(color='none', edgecolor='#918E88', ax=ax,
#             linewidth=2)

# nhd_sites[nhd_sites['WidthM'] >= 50].plot(ax=ax, color='dodgerblue', 
#                                         markersize=25, marker='o', 
#                                         # linewidth=0.7, edgecolor='k'
#                                          )
# nhd_sites[nhd_sites['WidthM'] < 50].plot(ax=ax, color='black',
#                                          markersize=19, marker='s',
#                                          # linewidth=0.7, edgecolor='grey'
#                                         )

# # # Create a black dot marker for Gauge
# # gauge_handle = Line2D([0], [0], marker='o', color='dodgerblue', markersize=10,
# #                       linestyle='None', label='Gauge',
# #                       markeredgecolor='black',
# #                         markeredgewidth=0.7)
# # gauge_handle_n = Line2D([0], [0], marker='s', color='black', markersize=8,
# #                         linestyle='None', label='Narrow gauge',
# #                         markeredgecolor='k',
# #                         markeredgewidth=0.7)

# # 1. Create the L-frame (Remove top/right)
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

# # 2. Apply the custom degree formatters
# ax.xaxis.set_major_formatter(FuncFormatter(lon_format))
# ax.yaxis.set_major_formatter(FuncFormatter(lat_format))

# # 3. Clean up ticks
# ax.xaxis.set_ticks_position('bottom')
# ax.yaxis.set_ticks_position('left')
# # ax.tick_params(labelsize=14)

# # plt.savefig(f'{save_dir}nhd_gauge_map.png', dpi=300, bbox_inches='tight')
# # plt.savefig(os.path.join(save_dir, 'nhd_map.svg'), format='svg', bbox_inches='tight')

# # ax.table(cellText=df.values, colLabels=df.columns, loc='bottom', bbox=[0, -0.3, 1, 0.2]) # Adjust bbox as needed
# # plt.subplots_adjust(bottom=0.2) # Adjust plot area to make room for table
# # plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(20,20))

# all_color = 'dodgerblue' #'cornflowerblue'
# filter_color = sns.color_palette("tab20b")[13]
# match_color = 'red'

# wbd.plot(color='#F5F0E5', ax=ax, edgecolor='white', linewidth=0.5)
# # wbd.plot(ax=ax, color='#E0E0E0', edgecolor='white', linewidth=0.5)

# states.plot(color='none', edgecolor='dimgray', ax=ax, linewidth=1)

# nhd_sites[(nhd_sites['WidthM'] >= 50)
#     & ~(nhd_sites['site_id'].isin(match_ids))
#     & ~(nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax, color=all_color, markersize=37,
#     marker='o', edgecolor='k', linewidth=0.7
# )

# nhd_sites[(nhd_sites['WidthM'] < 50)
#     & ~(nhd_sites['site_id'].isin(match_ids))
#     & ~(nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax, color=all_color, markersize=29,
#     marker='s', edgecolor='k', linewidth=0.7,
# )

# nhd_sites[(nhd_sites['WidthM'] >= 50)
#     & ~(nhd_sites['site_id'].isin(match_ids))
#     & (nhd_sites['site_id'].isin(filter_ids))
#     ].plot(
#     ax=ax, color=filter_color, markersize=37,
#     marker='o', edgecolor='k', linewidth=0.7
# )

# nhd_sites[(nhd_sites['WidthM'] < 50)
#     & ~(nhd_sites['site_id'].isin(match_ids))
#     & (nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax, color=filter_color, markersize=29,
#     marker='s', edgecolor='k', linewidth=0.7,
# )

# nhd_sites[(nhd_sites['WidthM'] >= 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
#     ax=ax, color=match_color, markersize=37,
#     marker='o', edgecolor='k', linewidth=0.7
# )
# # Narrow Rivers (< 50) + Match
# nhd_sites[(nhd_sites['WidthM'] < 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
#     ax=ax, color=match_color, markersize=29,
#     marker='s', edgecolor='k', linewidth=0.7,
# )


# L_MS = 7

# h_all = mlines.Line2D([], [], color=all_color, marker='o', linestyle='None',
#                       markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)

# h_filtered = mlines.Line2D([], [], color=filter_color, marker='o', linestyle='None',
#                            markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)

# h_match = mlines.Line2D([], [], color=match_color, marker='o', linestyle='None',
#                         markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)

# # 3. Create the Shape-only handles (Empty fill)
# h_wide = mlines.Line2D([], [], color='none', marker='o', linestyle='None',
#                        markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)

# h_narrow = mlines.Line2D([], [], color='none', marker='s', linestyle='None',
#                          markersize=L_MS-1, markeredgecolor='k', markeredgewidth=0.7)

# # 4. Create the blank space
# h_blank = mlines.Line2D([], [], linestyle='None', marker='', alpha=0)

# # 5. Assemble lists
# handles = [h_all, h_filtered, h_match, h_blank, h_wide, h_narrow]
# labels = [
#     'All sites', 
#     'Filtered sites', 
#     'Filtered sites in validation set',
#     '', 
#     'Rivers >= 50 m wide', 
#     'Rivers < 50 m wide'
# ]

# ax.legend(handles=handles, labels=labels,
#           loc='upper right', 
#           frameon=False,
#           fontsize=10, labelspacing=1.1)


# # 1. Create the L-frame (Remove top/right)
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

# # 2. Apply the custom degree formatters
# ax.xaxis.set_major_formatter(FuncFormatter(lon_format))
# ax.yaxis.set_major_formatter(FuncFormatter(lat_format))

# # 3. Clean up ticks
# ax.xaxis.set_ticks_position('bottom')
# ax.yaxis.set_ticks_position('left')
# # ax.tick_params(labelsize=14)

# # plt.savefig(f'{save_dir}nhd_gauge_map.png', dpi=300, bbox_inches='tight')
# # plt.savefig(os.path.join(save_dir, 'nhd_map.svg'), format='svg', bbox_inches='tight')

# # ax.table(cellText=df.values, colLabels=df.columns, loc='bottom', bbox=[0, -0.3, 1, 0.2]) # Adjust bbox as needed
# # plt.subplots_adjust(bottom=0.2) # Adjust plot area to make room for table
# # plt.show()

In [ ]:
# # 1. Create a figure with 1 row and 2 columns
# # Adjust figsize to be wider to accommodate both plots
# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 30))

# all_color = 'dodgerblue'
# filter_color = sns.color_palette("tab20b")[13]
# match_color = 'red'

# # --- LEFT SUBPLOT (ax1): AREA PLOT ---
# wbd.plot(color='#F5F0E5', ax=ax1, edgecolor='white', linewidth=0.5)
# states.plot(color='none', edgecolor='dimgray', ax=ax1, linewidth=1)

# # Regional overlays
# wbd_ne.plot(color=palette_letter_dict['NE'], ax=ax1, edgecolor='white', linewidth=0.5, label='NE')
# wbd_se.plot(color=palette_letter_dict['SE'], ax=ax1, edgecolor='white', linewidth=0.5, label='SE')
# wbd_sw.plot(color=palette_letter_dict['SW'], ax=ax1, edgecolor='white', linewidth=0.5, label='SW')
# wbd_nw.plot(color=palette_letter_dict['NW'], ax=ax1, edgecolor='white', linewidth=0.5, label='NW')

# # --- RIGHT SUBPLOT (ax2): SITES PLOT ---
# wbd.plot(color='#F5F0E5', ax=ax2, edgecolor='white', linewidth=0.5)
# states.plot(color='none', edgecolor='dimgray', ax=ax2, linewidth=1)

# # Plot markers on ax2
# # Wide Rivers (Circle)
# nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(filter_ids))].plot(ax=ax2, color=all_color, markersize=37, marker='o', edgecolor='k', linewidth=0.7)
# nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(filter_ids))].plot(ax=ax2, color=filter_color, markersize=37, marker='o', edgecolor='k', linewidth=0.7)
# nhd_sites[(nhd_sites['WidthM'] >= 50) & (nhd_sites['site_id'].isin(match_ids))].plot(ax=ax2, color=match_color, markersize=37, marker='o', edgecolor='k', linewidth=0.7)

# # Narrow Rivers (Square)
# nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(filter_ids))].plot(ax=ax2, color=all_color, markersize=29, marker='s', edgecolor='k', linewidth=0.7)
# nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(filter_ids))].plot(ax=ax2, color=filter_color, markersize=29, marker='s', edgecolor='k', linewidth=0.7)
# nhd_sites[(nhd_sites['WidthM'] < 50) & (nhd_sites['site_id'].isin(match_ids))].plot(ax=ax2, color=match_color, markersize=29, marker='s', edgecolor='k', linewidth=0.7)

# # --- SHARED FORMATTING ---
# L_MS = 7
# handles = [
#     mlines.Line2D([], [], color=all_color, marker='o', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], color=filter_color, marker='o', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], color=match_color, marker='o', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], linestyle='None', marker='', alpha=0),
#     mlines.Line2D([], [], color='none', marker='o', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], color='none', marker='s', linestyle='None', markersize=L_MS-1, markeredgecolor='k', markeredgewidth=0.7)
# ]
# labels = ['All sites', 'Filtered sites', 'Filtered sites in validation set', '', 'Rivers >= 50 m wide', 'Rivers < 50 m wide']

# ax2.legend(handles=handles, labels=labels, loc='upper right', frameon=False, fontsize=14, labelspacing=1.1)

# # Apply styling to both axes
# for ax in [ax1, ax2]:
#     # ax.set_aspect('equal')
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)
#     ax.tick_params(axis='both', which='major', labelsize=14)
#     ax.xaxis.set_major_formatter(FuncFormatter(lon_format))
#     ax.yaxis.set_major_formatter(FuncFormatter(lat_format))
#     ax.xaxis.set_ticks_position('bottom')
#     ax.yaxis.set_ticks_position('left')

# # plt.tight_layout(pad=0) # pad=0 removes almost all padding
# # plt.subplots_adjust(hspace=0.05) 
# plt.savefig(f'{save_dir}combined_map.png', dpi=300, bbox_inches='tight')

# # plt.tight_layout()
# # plt.show()

In [ ]:
# Create Figure 2

all_color = '#E8E8E8'
post_color = 'darkgray' 
match_color = 'deeppink'

fig2, ax2 = plt.subplots(figsize=(20, 20))

# --- SITES PLOT (ax2) ---
wbd.plot(
    color=#'#E0E0E0',
    '#E8E8E8',
    # '#DCDCDC',
    # '#F5F0E5', 
    ax=ax2, edgecolor='white', linewidth=0.5)
states.plot(color='none', edgecolor='dimgray', ax=ax2, linewidth=1)

# All sites
nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(post_ids))].plot(
    ax=ax2, color=all_color, markersize=59, marker='o',
    # edgecolor=(0,0,0,0.6),
    edgecolor='k',
    linewidth=0.7,
    # alpha=0.6
)
nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(post_ids))].plot(
    ax=ax2, color=all_color, markersize=51, marker='s',
    # edgecolor=(0,0,0,0.6),
    edgecolor='k',
    linewidth=0.7,
    # alpha=0.6
)

# Filtered sites
nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(post_ids))].plot(
    ax=ax2, color=post_color, markersize=59, marker='o', edgecolor='k', linewidth=0.7
)
nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(post_ids))].plot(
    ax=ax2, color=post_color, markersize=51, marker='s', edgecolor='k', linewidth=0.7
)


# Matched sites (validation set)
nhd_sites[(nhd_sites['WidthM'] >= 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
    ax=ax2, color=match_color, markersize=59, marker='o', edgecolor='k', linewidth=0.7
)
nhd_sites[(nhd_sites['WidthM'] < 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
    ax=ax2, color=match_color, markersize=51, marker='s', edgecolor='k', linewidth=0.7
)

# --- SHARED FORMATTING ---
L_MS = 10
handles = [
    mlines.Line2D([], [], color=all_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='silver', markeredgewidth=0.7),
    mlines.Line2D([], [], color=post_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
    mlines.Line2D([], [], color=match_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
    # mlines.Line2D([], [], linestyle='None', marker='', alpha=0),
    mlines.Line2D([], [], color='none', marker='o', linestyle='None', markersize=L_MS+1, markeredgecolor='k', markeredgewidth=0.7),
    mlines.Line2D([], [], color='none', marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)
]
labels = ['All sites', 'Sites with any data after pre-processing', 'Sites with any data in validation set',
          # '',
          'Rivers >= 50 m wide', 'Rivers < 50 m wide']


# Legend for Figure 2
ax2.legend(handles=handles, labels=labels, loc='upper right',
           frameon=False, fontsize=16)

# Formatting for Figure 2
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
# ax2.tick_params(axis='both', labelsize=14)
ax2.xaxis.set_major_formatter(FuncFormatter(lon_format))
ax2.yaxis.set_major_formatter(FuncFormatter(lat_format))

plt.savefig(f'{save_dir}sites_map.png', dpi=300, bbox_inches='tight')
# plt.savefig(os.path.join(save_dir, 'sites_map.svg'), format='svg', bbox_inches='tight')
# plt.close(fig2)

In [ ]:
# # Create Figure 2

# # all_color = 'lightskyblue'
# # filter_color = sns.color_palette("tab20b")[13]
# # match_color = '#ff7f0e' #'orange' #'mediumseagreen' #'#00FF7F' #'red'

# all_color = '#7570B3' #'#8DAE91' #'#D3D3D3'
# filter_color = '#8DAE91' #'lightskyblue' #'#56B4E9'
# match_color = 'deeppink' #'#FF4500' #'tomato'

# fig2, ax2 = plt.subplots(figsize=(20, 20))

# # --- SITES PLOT (ax2) ---
# wbd.plot(
#     color=#'#E0E0E0',
#     '#E8E8E8',
#     # '#DCDCDC',
#     # '#F5F0E5', 
#     ax=ax2, edgecolor='white', linewidth=0.5)
# states.plot(color='none', edgecolor='dimgray', ax=ax2, linewidth=1)

# # All sites
# nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax2, color=all_color, markersize=59, marker='o',
#     # edgecolor=(0,0,0,0.6),
#     edgecolor='silver',
#     linewidth=0.7,
#     # alpha=0.6
# )
# nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & ~(nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax2, color=all_color, markersize=51, marker='s',
#     # edgecolor=(0,0,0,0.6),
#     edgecolor='silver',
#     linewidth=0.7,
#     # alpha=0.6
# )

# # Filtered sites
# nhd_sites[(nhd_sites['WidthM'] >= 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax2, color=filter_color, markersize=59, marker='o', edgecolor='k', linewidth=0.7
# )
# nhd_sites[(nhd_sites['WidthM'] < 50) & ~(nhd_sites['site_id'].isin(match_ids)) & (nhd_sites['site_id'].isin(filter_ids))].plot(
#     ax=ax2, color=filter_color, markersize=51, marker='s', edgecolor='k', linewidth=0.7
# )


# # Matched sites (validation set)
# nhd_sites[(nhd_sites['WidthM'] >= 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
#     ax=ax2, color=match_color, markersize=59, marker='o', edgecolor='k', linewidth=0.7
# )
# nhd_sites[(nhd_sites['WidthM'] < 50) & (nhd_sites['site_id'].isin(match_ids))].plot(
#     ax=ax2, color=match_color, markersize=51, marker='s', edgecolor='k', linewidth=0.7
# )

# # --- SHARED FORMATTING ---
# L_MS = 10
# handles = [
#     mlines.Line2D([], [], color=all_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='silver', markeredgewidth=0.7),
#     mlines.Line2D([], [], color=filter_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], color=match_color, marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7),
#     # mlines.Line2D([], [], linestyle='None', marker='', alpha=0),
#     mlines.Line2D([], [], color='none', marker='o', linestyle='None', markersize=L_MS+1, markeredgecolor='k', markeredgewidth=0.7),
#     mlines.Line2D([], [], color='none', marker='s', linestyle='None', markersize=L_MS, markeredgecolor='k', markeredgewidth=0.7)
# ]
# labels = ['All sites', 'Filtered sites', 'Filtered sites in validation set',
#           # '',
#           'Rivers >= 50 m wide', 'Rivers < 50 m wide']


# # Legend for Figure 2
# ax2.legend(handles=handles, labels=labels, loc='upper right',
#            frameon=False, fontsize=16)

# # Formatting for Figure 2
# ax2.spines['top'].set_visible(False)
# ax2.spines['right'].set_visible(False)
# # ax2.tick_params(axis='both', labelsize=14)
# ax2.xaxis.set_major_formatter(FuncFormatter(lon_format))
# ax2.yaxis.set_major_formatter(FuncFormatter(lat_format))

# # plt.savefig(f'{save_dir}sites_map.png', dpi=300, bbox_inches='tight')
# # plt.savefig(os.path.join(save_dir, 'sites_map.svg'), format='svg', bbox_inches='tight')
# # plt.close(fig2)

## Analysis

In [ ]:
# Create merged dataframe of all flowlines intersected
# Initialize lists
d = []

# Loop through indices and store in lists
for i in indices_01:
    # Read prepped NHD
    flowlines, _, _ = readNHD(index=i)
    # Append to lists
    d.append(flowlines)
# Merge GeoDataFrames
flowlines_01 = pd.concat(d)

In [ ]:
# Create merged dataframe of all flowlines intersected
# Initialize lists
d = []

# Loop through indices and store in lists
for i in indices_03:
    # Read prepped NHD
    flowlines, _, _ = readNHD(index=i)
    # Append to lists
    d.append(flowlines)
# Merge GeoDataFrames
flowlines_03 = pd.concat(d)

In [ ]:
# Create merged dataframe of all flowlines intersected
# Initialize lists
d = []

# Loop through indices and store in lists
for i in indices_15:
    # Read prepped NHD
    flowlines, _, _ = readNHD(index=i)
    # Append to lists
    d.append(flowlines)
# Merge GeoDataFrames
flowlines_15 = pd.concat(d)

In [ ]:
# Create merged dataframe of all flowlines intersected
# Initialize lists
d = []

# Loop through indices and store in lists
for i in indices_17:
    # Read prepped NHD
    flowlines, _, _ = readNHD(index=i)
    # Append to lists
    d.append(flowlines)
# Merge GeoDataFrames
flowlines_17 = pd.concat(d)

In [ ]:
flowlines_01['huc2'] = '01'
flowlines_03['huc2'] = '03'
flowlines_15['huc2'] = '15'
flowlines_17['huc2'] = '17'

In [ ]:
flowlines_01[flowlines_01['WidthM'] > 10].shape[0]

In [ ]:
flowlines_03[flowlines_03['WidthM'] > 10].shape[0]

In [ ]:
flowlines_15[flowlines_15['WidthM'] > 10].shape[0]

In [ ]:
flowlines_17[flowlines_17['WidthM'] > 10].shape[0]

In [ ]:
flowlines_01[flowlines_01['WidthM'] > 10].shape[0] +\
flowlines_03[flowlines_03['WidthM'] > 10].shape[0] +\
flowlines_15[flowlines_15['WidthM'] > 10].shape[0] +\
flowlines_17[flowlines_17['WidthM'] > 10].shape[0]

In [ ]:
flowlines_01.shape[0]

In [ ]:
flowlines_03.shape[0]

In [ ]:
flowlines_15.shape[0]

In [ ]:
flowlines_17.shape[0]

In [ ]:
data = {
    'NE': [flowlines_01.shape[0]],
    'SE': [flowlines_03.shape[0]],
    'SW': [flowlines_15.shape[0]],
    'NW': [flowlines_17.shape[0]]
}

df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
temp_01 = flowlines_01[['StreamOrde', 'Bin', 'huc2']]
temp_03 = flowlines_03[['StreamOrde', 'Bin', 'huc2']]
temp_15 = flowlines_15[['StreamOrde', 'Bin', 'huc2']]
temp_17 = flowlines_17[['StreamOrde', 'Bin', 'huc2']]

In [ ]:
all_ = pd.concat(objs=[temp_01, temp_03, temp_15, temp_17]).reset_index().drop(columns='index')

In [ ]:
all_['Bin'] = pd.Categorical(all_['Bin'], order)

In [ ]:
huc2_order = ['17','03','15','01']
all_['huc2'] = pd.Categorical(all_['huc2'], categories=huc2_order, ordered=True)

In [ ]:
fig, ax = plt.subplots(figsize=(16,10))
sns.set(style='white')

sns.histplot(data=all_, x='StreamOrde',
             hue='huc2', palette=palette_dict,
             multiple='dodge',
             discrete=True,
             shrink=0.8,
             # edgecolor='None',
             # legend='auto',
             ax=ax)

legend_labels = ['NE', 'SE', 'SW', 'NW']
legend_colors = [palette_letter_dict[h] for h in legend_labels]
handles = [Patch(facecolor=color, label=label) for color, label in zip(legend_colors, legend_labels)]

ax.legend(handles=handles, title='Region', fontsize=16,
          title_fontsize=16);
plt.xlabel('Stream Order', fontsize=16,
           fontweight='bold', labelpad=20);
plt.ylabel('Number of reaches (log)', fontsize=16,
           fontweight='bold');
plt.yscale('log')

plt.xticks(np.arange(0,12), fontsize=14, fontweight='semibold');
plt.yticks(fontsize=14, fontweight='semibold');

plt.savefig(f'{save_dir}nhd_stream_order.png', dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(16,10))
sns.set(style='white')

sns.histplot(data=all_, x='Bin',
             hue='huc2', palette=palette_dict,
             multiple='dodge',
             discrete=True,
             shrink=0.8,
             # edgecolor='None',
             # legend='auto',
             ax=ax)

# Create custom legend handles using your palette
legend_labels = ['NE', 'SE', 'SW', 'NW']
legend_colors = [palette_letter_dict[h] for h in legend_labels]
handles = [Patch(facecolor=color, label=label) for color, label in zip(legend_colors, legend_labels)]

ax.legend(handles=handles, title='Region', fontsize=16,
          title_fontsize=16);
plt.xlabel('Width (m)', fontsize=16,
           fontweight='bold', labelpad=20);
plt.ylabel('Number of reaches (log)', fontsize=16,
           fontweight='bold');
plt.yscale('log')

plt.xticks(np.arange(0,14), rotation=-45, fontsize=14, fontweight='semibold');
plt.yticks(fontsize=14, fontweight='semibold');

plt.savefig(f'{save_dir}nhd_width.png', dpi=300, bbox_inches='tight')